# Notebook 5: Interactive visualization of subgraphs

Import the Python library dependencies.

In [1]:
import watermark

from yfiles_jupyter_graphs_for_kuzu import KuzuGraphWidget
import kuzu
#import ryu

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-12T13:04:55.332378-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

kuzu                          : 0.9.0
watermark                     : 2.5.0
yfiles_jupyter_graphs_for_kuzu: 0.0.4



## yFiles Interactive Visualization

Reconnect to the existing graph database.

In [3]:
DB_PATH: str = "./db"

db: kuzu.Database = kuzu.Database(DB_PATH)
conn: kuzu.Connection = kuzu.Connection(db)

#db: ryu.Database = ryu.Database(DB_PATH)
#conn: ryu.Connection = ryu.Connection(db)

Define a function to adjust the node and edge configurations, to help clarify the visualization.
This uses [Viz Palette](https://projects.susielu.com/viz-palette).

In [4]:
def node_color (
    node: dict
    ) -> str:
    match node["properties"]["class"]:
        case "sz:Person":
            return "#B8C25F"
        case "sz:Organization":
            return "#8C6894"
        case "sz:DataRecord":
            return "#B7B38E"
        case _:
            return "#000000"

Define a function to customize the node scaling -- within a range of `(0.0, 2.0]` -- based on the computed centrality measures.

In [5]:
def node_scale_factor (
    node: dict,
    base: float = 0.8,
    rate: float = 20.0,
    ) -> float:
    centrality: float = node["properties"]["betweenness_centrality"]
    limit: float = 2.0 - base
    curve: float = limit * (rate**centrality - 1.0) / max(0.01, (rate - 1.0))
    
    return min(
        2.0,
        curve + base,
    )

Now create a [`yFiles`](https://www.yworks.com/products/yfiles-graphs-for-jupyter) graph widget to explore the subgraphs.

In [6]:
widget: KuzuGraphWidget = KuzuGraphWidget(conn)

# design for nodes

widget.add_node_configuration(
    "Entity",
    color = node_color,
    text = lambda node: {"text": node["properties"]["descrip"]},
    scale_factor = node_scale_factor,
)

widget.add_node_configuration(
    "OpenSanctions",
    color = "#D9CAD7",
    text = lambda node: {"text": node["properties"]["descrip"]},
    scale_factor = node_scale_factor,
)

widget.add_node_configuration(
    "OpenOwnership",
    color = "#B8C25F",
    text = lambda node: {"text": node["properties"]["descrip"]},
    scale_factor = node_scale_factor,
)

widget.add_node_configuration(
    "Risk",
    color = "#C25FB8",
    text = lambda node: {"text": node["properties"]["topic"]},
    size = (23, 23),
)

# design for edges

widget.add_relationship_configuration(
    "Related",
    color = "#949068",
    text = lambda edge: {"text": edge["properties"]["sem_rel"]},
)

widget.add_relationship_configuration(
    "Matched",
    color = "#D9CAD7",
    text = lambda edge: {"text": edge["properties"]["sem_rel"]},
)

widget.add_relationship_configuration(
    "Role",
    color = "#C25FB8",
    text = lambda edge: {"text": edge["properties"]["role"]},
)

widget.add_relationship_configuration(
    "HasRisk",
    color = "#C25FB8",
)

Run an interactive visualization using `yFiles` to examine the subgraph describing the [2021 South London Papa Johns](https://www.newsshopper.co.uk/news/19164815.boss-bromley-catford-papa-johns-stores-jailed/) tax evasion case.

As you scroll over different graph elements, notice how the design of the interactive visualization highlights important details for investigators to notice:

 - tool-tips show the properties of nodes and edges, i.e., information collected into the KG
 - relative node size indicates the scaled _betweenness centrality_ measures
 - the `evidence` property on edges shows the _entity resolution_ matching criteria, which includes _ultimate benefial ownership_ (UBO) information from Open Ownership
 - the `Risk` nodes show known risks from OpenSanctions about the entities to which they are connected

In [7]:
query: str = """
MATCH (a)-[b]->(c:Entity)-[d *1..5]->(e)
WHERE c.descrip CONTAINS "Abassin"
RETURN * LIMIT 200;
"""

widget.show_cypher(
    query,
    layout = "radial"
)

GraphWidget(layout=Layout(height='500px', width='100%'))

In particular, notice how [Abassin Badshah](https://qsrmedia.co.uk/legal/more-news/papa-johns-franchisee-jailed-tax-fraud-report) is the most prominant node in this subgraph, with beneficial ownership of four shell companies involved in the tax evasion case, plus a risk noted from [UK Companies House](https://find-and-update.company-information.service.gov.uk/disqualified-officers/natural/mGquuTbmESWiRmHJPz1ObUwfDgk) for _corporate disqualification_.

Let's see how the same results look as a dataframe.

In [8]:
conn.execute(query).get_as_pl()

a,c,b,e,d
struct[10],struct[6],struct[9],struct[10],struct[2]
"{{198,0},""Entity"",""sz:99"",""WELLHANCIA HEALTH CARE LTD"",""sz:Organization"",0.011279,null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person"",0.073316}","{{198,0},{4,0},""Related"",{406,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OOR(APPOINTMENT_OF_BOARD,SHAREHOLDING 75% 100%,VOTING_RIGHTS 75% 100%:)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,null,""corp.disqual""}","{[{{18,0},""Entity"",null,""sz:156"",""Rehana Badshah"",""sz:Person"",null,null,0.010715,null}, {{4,0},""Entity"",null,""sz:1"",""Abassin Badshah"",""sz:Person"",null,null,0.073316,null}, {{0,2},""OpenSanctions"",null,""sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ"",""Abassin BADSHAH"",""sz:Person"",""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",""https://www.opensanctions.org/entities/NK-25vyVFzt8vdJGgAXMRTwTJ"",0.0,null}],[{{4,0},{18,0},""Related"",{887,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}, {{18,0},{4,0},""Related"",{942,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}, … {{0,2},{2,4},""HasRisk"",{0,6},null,null,null,null,null}]}"
"{{126,0},""Entity"",""sz:2"",""LMAR GB LTD"",""sz:Organization"",0.011279,null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person"",0.073316}","{{126,0},{4,0},""Related"",{707,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(:DIRECTORSHIP)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,null,""corp.disqual""}","{[{{18,0},""Entity"",null,""sz:156"",""Rehana Badshah"",""sz:Person"",null,null,0.010715,null}, {{4,0},""Entity"",null,""sz:1"",""Abassin Badshah"",""sz:Person"",null,null,0.073316,null}, {{0,2},""OpenSanctions"",null,""sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ"",""Abassin BADSHAH"",""sz:Person"",""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",""https://www.opensanctions.org/entities/NK-25vyVFzt8vdJGgAXMRTwTJ"",0.0,null}],[{{4,0},{18,0},""Related"",{887,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}, {{18,0},{4,0},""Related"",{942,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}, … {{0,2},{2,4},""HasRisk"",{0,6},null,null,null,null,null}]}"
"{{193,0},""Entity"",""sz:9"",""BARLLOWS SERVICES LTD"",""sz:Organization"",0.011279,null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person"",0.073316}","{{193,0},{4,0},""Related"",{728,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(:DIRECTORSHIP)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,null,""corp.disqual""}","{[{{18,0},""Entity"",null,""sz:156"",""Rehana Badshah"",""sz:Person"",null,null,0.010715,null}, {{4,0},""Entity"",null,""sz:1"",""Abassin Badshah"",""sz:Person"",null,null,0.073316,null}, {{0,2},""OpenSanctions"",null,""sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ"",""Abassin BADSHAH"",""sz:Person"",""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",""https://www.opensanctions.org/entities/NK-25vyVFzt8vdJGgAXMRTwTJ"",0.0,null}],[{{4,0},{18,0},""Related"",{887,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}, {{18,0},{4,0},""Related"",{942,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}, … {{0,2},{2,4},""HasRisk"",{0,6},null,null,null,null,null}]}"
"{{104,0},""Entity"",""sz:155"",""BARLLOWS SERVICES LTD"",""sz:Organization"",0.011279,null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person"",0.073316}","{{104,0},{4,0},""Related"",{926,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OOR(SHAREHOLDING 50% 75%,VOTING_RIGHTS 50% 75%:)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,null,""corp.disqual""}","{[{{18,0},""Entity"",null,""sz:156"",""Rehana Badshah"",""sz:Person"",null,null,0.010715,null}, {{4,0},""Entity"",null,""sz:1"",""Abassin Badshah"",""sz:P

## Design for Human Scale

Consider the side-by-side comparison of using of an interactive visualization versus using a table of query results, although admittedly the table could be formatted much better!

In general the effectiveness of the visualization builds on the results of graph algorithms, plus the _risk_ and _UBO_ information connected into the graph through _entity resolution_. This is a realization of the **four-step design pattern** mentioned earlier.

Rather than trying to visualize "billions and billions" of elements in some ginormous graph, the people using these kinds of investigative graphs need to focus on smaller subgraphs -- for example, less than a dozen nodes in the case examined above. This is true whether the investigation is about [_hunting bad guys_](https://en.wikipedia.org/wiki/Bringing_Down_the_House_(book)) or about trying to find your best customers within your data.

Investigative graphs in industry and government tend to rely on people using _case management tools_, so we do this work to reshape subgraphs of potential interest, running each through case management. Moreover, it's clear that the important tools in this process are _entity resolution_, _computable semantics_, _graph analytics_, and effective _design for human scale_. 

The latter point in particular -- about design, UX, UI -- has become a bottleneck for the adoption of AI applications in production use. Ironically, the converse of that problem is where we leverage AI technologies to "refocus the lens" so that investigative teams can collaborate more effectively. Whereas the criminal tradecraft plus the enduring problems of data quality in enterprise both cause more [_complexity_](https://cynefin.io/wiki/Field_guide_to_managing_complexity_(and_chaos)_in_times_of_crisis) in these kinds of use cases, here in this tutorial we're showing production-quality approaches for "refocusing the lens". In other words, the intent is to bring inhumanly large, complex problems back to _human scale_. This is largely a design problem, which is becoming an AI problem.

_Personal note:_ the author focused on AI as grad student, taking an extra year during a teaching fellowship at Stanford to participate in the Design program -- which built atop optimization, lingusitics, software engineering, distributed systems, etc. The intent was to explore how to blend **Artificial Intelligence** technologies plus **Design for Human Scale** together. _Mas o menos_, this is a thesis statement.

Recommended resources about **Design for Human Scale**:

_Design for Human Scale_  
**Victor Papanek**  
(1981)  
<https://www.goodreads.com/en/book/show/2845800-design-for-human-scale>

_Humanscale_  
**Alvin Tilley**, **Joan Bardagjy**  
(1991)  
<https://www.goodreads.com/book/show/1725112.Humanscale>

_Understanding Computers and Cognition: A New Foundation for Design_  
**Terry Winograd**, **Fernando Flores**  
(1986)  
<https://www.goodreads.com/book/show/53482.Understanding_Computers_and_Cognition>

_Conversations For Action and Collected Essays: Instilling a Culture of Commitment in Working Relationships_  
**Fernando Flores**, **Maria Flores Letelier**  
(2013)  
<https://www.goodreads.com/book/show/17870114-conversations-for-action-and-collected-essays>

---

Finally, close the database connection.

In [9]:
db.close()

---